# 08 Production Hardening and Release Gates (OpenClaw, 2026)

## What This Lesson Is
Operationalize OpenClaw deployments with concrete release gates across security, reliability, and cost.

## Scientific Lens
- Concept: Production readiness requires objective gate criteria and rollback plans, not ad hoc confidence.
- Measure: Gate pass rate across weighted readiness dimensions.
- Validity Limit: Passing gates reduces risk but cannot remove uncertainty in live distributed systems.


## How It Works
1. Define weighted release gates covering model policy, auth, observability, and incident readiness.
2. Compute deterministic go/no-go score and produce auditable decision artifact.
3. Collect live gateway/model snapshots and attach to release evidence bundle.


In [ ]:
import os
import shutil

HAS_OPENCLAW = shutil.which("openclaw") is not None
print("openclaw available:", HAS_OPENCLAW)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL", "<unset>"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: uses real OpenClaw integration commands and skips gracefully if prerequisites are missing.


In [ ]:
# Deterministic Demo
gates = {
    "model_policy_reviewed": (1.0, 2),
    "fallbacks_configured": (1.0, 2),
    "webhook_auth_enforced": (1.0, 2),
    "cron_runbook_present": (0.5, 1),
    "channel_pairing_hardened": (0.5, 1),
    "cost_guardrails_enabled": (0.5, 2),
}
weighted = sum(v*w for (v,w) in gates.values()) / sum(w for (_,w) in gates.values())
decision = "approve" if weighted >= 0.8 else "block"
print("score=", weighted, "decision=", decision)
assert decision in {"approve", "block"}


In [ ]:
# Live Demo
import json, shutil, subprocess

if not HAS_OPENCLAW:
    print("Skipping live hardening demo: openclaw CLI not installed.")
else:
    capture = {}
    for name, cmd in {
        "status": ["openclaw", "status"],
        "models_status": ["openclaw", "models", "status"],
        "cron_status": ["openclaw", "cron", "status"],
    }.items():
        p = subprocess.run(cmd, capture_output=True, text=True)
        capture[name] = (p.stdout or p.stderr).strip()[:1200]
    print(json.dumps(capture, indent=2))


## Applied Labs
1. Add a mandatory gate for evidence of webhook auth header enforcement.
2. Create an emergency rollback checklist and map each step to owner + SLA.
3. Build a release report template that embeds live snapshot excerpts and deterministic score.

## Validation Checklist
- Go/no-go decision is computed from explicit weighted gates.
- Release evidence includes both deterministic scoring and live platform state.
- Operational controls are tied to measurable criteria, not vague best-effort checks.

## Further Reading
- NIST AI RMF: https://www.nist.gov/itl/ai-risk-management-framework
- NIST SSDF: https://csrc.nist.gov/Projects/ssdf
- OpenClaw docs directory (ops+safety): https://docs.openclaw.ai/start/docs-directory
